In [17]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import roc_auc_score

In [18]:
dataset = load_dataset("katielink/moleculenet-benchmark", "tox21", split="train")

In [19]:
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_subset = split_dataset["train"]
eval_subset = split_dataset["test"]

In [20]:
label_columns = [col for col in dataset.column_names if col not in ["smiles", "mol_id"]]
print(f"Loaded {len(train_subset)} training samples, {len(eval_subset)} validation samples across {len(label_columns)} tasks.")

Loaded 5011 training samples, 1253 validation samples across 12 tasks.


In [21]:
def process_data(subset):
    smiles = [str(x) for x in subset["smiles"]]
    labels = np.array(
        [[float(row[col]) if row[col] is not None and not (isinstance(row[col], str) and row[col].strip() == '') else 0.0
          for col in label_columns]
         for row in subset],
        dtype=np.float32
    )
    return smiles, labels

train_smiles, train_labels = process_data(train_subset)
eval_smiles, eval_labels = process_data(eval_subset)

In [22]:
model_checkpoint = "seyonec/PubChem10M_SMILES_BPE_450k"
print(f"Tokenizing datasets using {model_checkpoint}...")
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

train_encodings = tokenizer(train_smiles, truncation=True, padding=True, max_length=128)
eval_encodings = tokenizer(eval_smiles, truncation=True, padding=True, max_length=128)

Tokenizing datasets using seyonec/PubChem10M_SMILES_BPE_450k...


In [23]:
class MolecularDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = MolecularDataset(train_encodings, train_labels)
eval_dataset = MolecularDataset(eval_encodings, eval_labels)

In [24]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Apply sigmoid since it's multi-label binary classification
    preds = 1 / (1 + np.exp(-logits))

    auc_scores = []
    for i in range(labels.shape[1]):
        # Calculate ROC-AUC per task, ignoring columns with only one class value
        try:
            score = roc_auc_score(labels[:, i], preds[:, i])
            auc_scores.append(score)
        except ValueError:
            continue

    mean_auc = np.mean(auc_scores) if auc_scores else 0.0
    return {"roc_auc": mean_auc}

In [25]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_columns),
    problem_type="multi_label_classification"
)

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: seyonec/PubChem10M_SMILES_BPE_450k
Key                         | Status     | 
----------------------------+------------+-
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstre

In [30]:
training_args = TrainingArguments(
    output_dir='./chemberta-scaled-results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100,
    learning_rate=3e-5,
    weight_decay=0.01
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

In [31]:
trainer.train()

Epoch,Training Loss,Validation Loss,Roc Auc
1,0.207492,0.198941,0.742694
2,0.183527,0.184422,0.794350
3,0.166564,0.180773,0.803305


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=942, training_loss=0.19382722362591204, metrics={'train_runtime': 239.0033, 'train_samples_per_second': 62.899, 'train_steps_per_second': 3.941, 'total_flos': 497934384694272.0, 'train_loss': 0.19382722362591204, 'epoch': 3.0})

In [32]:
trainer.save_model("./chemberta-tox21-finetuned")
tokenizer.save_pretrained("./chemberta-tox21-finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./chemberta-tox21-finetuned/tokenizer_config.json',
 './chemberta-tox21-finetuned/tokenizer.json')